# TP2 - Informe tecnico

Sistema de Deteccion y Clasificacion de Razas de Perros — IA 5.2 Computer Vision.

## Equipo
- Alumno 1: Agustín Accurso
- Alumno 2: Lautaro Cena

## 1. Explicacion completa del pipeline

TO-DO: describir el flujo Embeddings -> Busqueda por similitud -> Clasificacion -> Deteccion -> Pipeline completo,
y como se integran los componentes (base vectorial, modelos, YOLO, aplicacion Gradio).

## 2. Dataset

TO-DO: distribucion de clases, cantidad de imagenes por raza, definicion de splits
(train/valid/test) y conjunto independiente de evaluacion.

## 3. Preprocesamiento

TO-DO: tecnicas aplicadas (resize, normalizacion, data augmentation, filtrado) y su justificacion.

## 4. Justificacion de los modelos elegidos

TO-DO: modelo de embeddings baseline (Etapa 1), ResNet18 fine-tuned y CNN custom (Etapa 2),
YOLO (Etapa 3). Analizar trade-offs: precision, velocidad de inferencia, consumo de memoria
y complejidad computacional.

## 5. Proceso de entrenamiento e hiperparametros

TO-DO: proceso de fine-tuning, hiperparametros utilizados (learning rate, batch size, epochs,
optimizador, scheduler), curvas de entrenamiento.

## 6. Resultados obtenidos

TO-DO:
- Etapa 1: NDCG@10 y justificacion del resultado.
- Etapa 2: accuracy, precision, recall, specificity, F1, matriz de confusion.

## 7. Comparacion entre enfoques

TO-DO: busqueda por similitud vs clasificacion supervisada; ResNet18 fine-tuned vs CNN custom.

## 8. Problemas encontrados y soluciones implementadas

TO-DO.

## 9. Modificaciones fuera de las funciones indicadas

TO-DO: justificar debidamente cualquier cambio realizado fuera de las funciones
indicadas en cada etapa (si no hubo, indicarlo).

**Nota sobre `data/embeddings.json` y `models/*.pth`:**

Estos archivos no se versionan en git (ver `.gitignore`) por ser
artefactos generados, regenerables ejecutando los scripts
correspondientes (`scripts/build_index.py` para la base vectorial,
`scripts/train_classifier.py` para los checkpoints de modelos), y por
su tamaño (la base vectorial completa supera varios MB con ~8000
embeddings de 512 dimensiones). Para reproducir el estado del sistema
desde cero: `python scripts/download_dataset.py`, seguido de
`python scripts/build_index.py --split train` y
`python scripts/train_classifier.py --model resnet18_finetuned`.

### Modificación justificada: `PgVectorEmbeddingStore.search()`

**Archivo:** `src/lib/storage/pgvector_store.py`

**Problema encontrado:** al ejecutar `search_similar_images` con el backend
pgvector, la consulta SQL fallaba con el error:

**Causa:** el operador `<=>` de la extensión pgvector requiere que ambos
operandos sean del tipo `vector`. El parámetro de la consulta (`%s`) se
pasa desde Python como una lista de floats, que psycopg serializa como
`double precision[]` (un array genérico de PostgreSQL) en lugar de
`vector`, a pesar de que `register_vector(self.conn)` se invoca en el
constructor de la clase. PostgreSQL no realiza el cast implícito
necesario para el operador `<=>` en este contexto.

**Solución aplicada:** se agregó un cast explícito a `vector` en la
clausula `ORDER BY` de la consulta:

```sql
-- Antes:
ORDER BY embedding <=> %s
-- Despues:
ORDER BY embedding <=> %s::vector
```

**Alcance de la modificación:** un único cast de tipo en una consulta
SQL, sin alterar la firma del metodo, su comportamiento esperado, ni
la interfaz `EmbeddingStoreProtocol`. No afecta a `extract_embedding`,
`search_similar_images` ni `predict_breed_from_neighbors` (funciones de
Etapa 1 implementadas por el estudiante), que permanecen sin cambios.

### Modificación justificada: `SimilarityService.__init__`

**Archivo:** `src/lib/services/similarity_service.py`

**Causa:** las tres funciones de Etapa 1 a implementar
(`extract_embedding`, `search_similar_images`,
`predict_breed_from_neighbors`) requieren acceso a un modelo de
PyTorch cargado, un dispositivo de cómputo (CPU/GPU) y una pipeline de
transformaciones de imagen. El constructor original solo almacenaba
los parámetros de configuración recibidos (`store`,
`similarity_metric`, `top_k`, etc.), sin inicializar ningún modelo.

**Solución aplicada:** se agregó al `__init__` la carga del modelo
baseline (ResNet18 pre-entrenado en ImageNet, sin la capa `fc`), la
selección de `device` (GPU si está disponible, CPU en caso contrario)
y la definición de las transformaciones de preprocesamiento (resize,
conversión a tensor, normalización con estadísticas de ImageNet). Se
cargan una sola vez al instanciar la clase, no en cada llamada a
`extract_embedding`, para evitar el costo repetido de cargar el modelo
en cada imagen procesada.

**Alcance de la modificación:** se agregan atributos nuevos
(`self.device`, `self._embedding_model`, `self._transform`) sin
eliminar ni alterar ninguno de los atributos ni parámetros originales
del constructor. La firma de `__init__` (parámetros que recibe) no
cambia. No afecta la interfaz pública de la clase ni el contrato que
usan `bootstrap.py` o los scripts (`build_index.py`,
`train_classifier.py`) al instanciarla.

### Gestión del repositorio

Se agregaron al archivo `.gitignore`:
 `data/embeddings.json`
y `data/external_eval/` del control de versiones. Ambos son artefactos
generados o descargados localmente (el índice de embeddings se
reconstruye con `scripts/build_index.py`, y el conjunto externo se
descarga con la celda correspondiente del notebook), por lo que
versionarlos no aporta valor y aumenta innecesariamente el tamaño del
repositorio.

### Cambios de infraestructura para Etapa 2 (fuera de train_classifier/evaluate_classifier)

**`config.py` (clase `Settings`)**
Se agregaron 7 campos nuevos: `batch_size`, `max_epochs`, `patience`,
`lr_head`, `lr_backbone`, `step_size`, `gamma`. Se definieron como
variables de entorno (con valores por defecto) siguiendo el mismo
patron ya usado en el proyecto para `RESNET18_MODEL_NAME` y demas
configuracion, en vez de hardcodear los hiperparametros de
entrenamiento dentro de `classifier_service.py`.

**`.env`**
Se agregaron las 7 variables correspondientes a los campos anteriores,
con los valores elegidos para el entrenamiento (detallados y
justificados en la seccion de Modelo A/B del informe).

**`bootstrap.py` (`build_classifier`)**
Se agrego el parametro `settings=settings` al construir
`ClassifierService`, para que el servicio tenga acceso a los
hiperparametros de `Settings` sin necesidad de pasarlos uno por uno
como argumentos sueltos del constructor.

**`classifier_service.py` — constructor (`__init__`)**
Se agrego el parametro `settings: Settings` y el atributo
`self.settings`, por el motivo anterior. Se uso un import bajo
`TYPE_CHECKING` para evitar un import circular entre
`classifier_service.py` y `config.py` (este ultimo no necesita conocer
`ClassifierService`).

**`classifier_service.py` — helpers nuevos**
Se agregaron dos metodos privados, `_build_transforms` y
`_build_resnet18_finetuned` (mas `_build_cnn_custom` si se implementa
el Modelo B), reutilizados por `train_classifier` y
`evaluate_classifier`. Esto evita que ambas funciones definan la
arquitectura del modelo y las transformaciones de forma independiente,
lo que podria desincronizarlas si se modifica una sin actualizar la
otra (por ejemplo, un cambio en que capas se descongelan para el
fine-tuning). No se factorizo la construccion del optimizador ni la
seleccion de modelo en helpers adicionales porque cada una se usa en
un unico punto del codigo (dentro de `train_classifier`), por lo que
separarlas no evitaba duplicacion real.